# 001 Claude Code Harness Engineering

这是 Harness Engineering 教学目录的第一份 Notebook。

学习目标：

1. 理解 Claude Code 这类编码代理的核心交互模式
2. 理解为什么“模型更聪明”不等于“系统更可靠”
3. 学会把 Prompt、Query Loop、Tool、Context、Recovery、Sub-agent、Team Policy 拆成工程组件
4. 用小型 Python 示例模拟可迁移的约束框架

本 Notebook 基于你给出的 Claude Code 设计提纲来教学。这里不声称逐行复现 Claude Code 源码，而是从源码阅读视角提炼可迁移的工程设计原则。

## 一句话定位

**Prompt 决定模型怎么说话，Harness 决定模型怎么做事。**

编码代理能执行命令、修改文件、调用工具。能力越强，事故半径越大。

所以核心问题不是：

- 模型会不会做？

而是：

- 它有没有权限做？
- 做之前有没有上下文预算？
- 做错了能不能恢复？
- 做完以后能不能独立验证？
- 中断以后能不能继续工作？

## Harness Engineering 是什么

这里把 Harness 理解成一套持续生效的控制结构。

它不是简单的安全壳，也不是一段更长的 system prompt。

它至少包含：

1. Prompt 分层拼装
2. Query Loop 状态机
3. 工具调度与权限裁决
4. 上下文预算与压缩
5. 错误恢复与熔断
6. 多代理隔离与生命周期管理
7. 团队制度、审计和验证标准

把模型当成不稳定部件，Harness 的职责就是让这个不稳定部件在工程边界内持续产出。

## 第一原则：Prompt 是控制面，不是人格装饰

很多人写 Prompt 时只关注“语气”和“角色”。

在编码代理里，Prompt 更像控制面配置：

- 默认系统规则
- 项目规则
- 用户自定义规则
- 本轮追加约束
- 哪些内容可缓存，哪些内容必须每轮重新计算

Prompt 不只是告诉模型“你是谁”，而是告诉模型“在这个系统里你能做什么、不能做什么、遇到冲突按什么优先级处理”。

In [ ]:
from dataclasses import dataclass


@dataclass
class PromptBlock:
    name: str
    priority: int
    content: str
    cacheable: bool


def assemble_prompt(blocks: list[PromptBlock]) -> str:
    ordered = sorted(blocks, key=lambda block: block.priority)
    return '\n\n'.join(f'[{block.name}]\n{block.content}' for block in ordered)


blocks = [
    PromptBlock('default-system', 10, '遵守安全边界，解释关键动作。', True),
    PromptBlock('project-rules', 20, '不要重置用户未提交的代码。', True),
    PromptBlock('session-task', 40, '新增一个教学 notebook。', False),
    PromptBlock('runtime-note', 50, '先验证 JSON，再结束任务。', False),
]

print(assemble_prompt(blocks))
print('cacheable blocks =', [block.name for block in blocks if block.cacheable])

## 第二原则：Query Loop 是代理系统的心跳

编码代理不是一次问答。

它通常是一个持续循环：

```text
接收用户输入
  -> 整理消息和上下文预算
  -> 调模型
  -> 消费流式事件
  -> 调工具
  -> 记录结果
  -> 判断继续、完成、中断或恢复
```

Query Loop 决定执行秩序。模型只是在这个循环里的一个不稳定推理部件。

In [ ]:
from enum import Enum


class LoopState(str, Enum):
    READY = 'ready'
    THINKING = 'thinking'
    TOOL_RUNNING = 'tool_running'
    INTERRUPTED = 'interrupted'
    RECOVERING = 'recovering'
    DONE = 'done'


ledger = []
state = LoopState.READY

def transition(next_state: LoopState, reason: str) -> None:
    global state
    ledger.append({'from': state.value, 'to': next_state.value, 'reason': reason})
    state = next_state


transition(LoopState.THINKING, '收到用户任务，开始推理')
transition(LoopState.TOOL_RUNNING, '模型请求读取文件')
transition(LoopState.INTERRUPTED, '用户中断')
transition(LoopState.RECOVERING, '补齐工具账本并恢复上下文')
transition(LoopState.DONE, '恢复后完成回答')

ledger

## 第三原则：工具是受管执行接口，权限是基本器官

模型可以建议调用工具，但不能天然拥有执行权。

一个可靠 Harness 至少要回答：

- 这个工具能不能并发？
- 这个工具会不会改文件？
- 这个命令是不是高危？
- 当前策略是 allow、deny，还是 ask？
- 工具失败后账本是否完整？

权限不是附加功能，而是编码代理的基本器官。

In [ ]:
from dataclasses import dataclass
from typing import Literal


Decision = Literal['allow', 'deny', 'ask']


@dataclass
class ToolCall:
    name: str
    command: str
    writes_files: bool = False
    destructive: bool = False


def decide_permission(call: ToolCall) -> Decision:
    if call.destructive:
        return 'ask'
    if call.name == 'bash' and 'rm -rf' in call.command:
        return 'deny'
    if call.writes_files:
        return 'ask'
    return 'allow'


tool_calls = [
    ToolCall('read', 'sed -n 1,120p README.md'),
    ToolCall('edit', 'update notebook index', writes_files=True),
    ToolCall('bash', 'rm -rf .venv', destructive=True),
]

[(call.command, decide_permission(call)) for call in tool_calls]

## 第四原则：上下文是工作内存，治理即预算治理

上下文不是越多越好。

可靠 Harness 会把上下文分层：

- 长期规则：类似 `CLAUDE.md`，放稳定约束
- 长期记忆：类似 `MEMORY.md`，放索引和沉淀，不放全文垃圾
- 会话记忆：当前任务摘要、计划、关键文件
- 临时对话：本轮交互细节

当预算紧张时，应该压缩临时内容，而不是丢失规则和任务语义。

In [ ]:
@dataclass
class ContextItem:
    layer: str
    text: str
    tokens: int
    must_keep: bool = False


def compact_context(items: list[ContextItem], budget: int) -> list[ContextItem]:
    kept = []
    used = 0
    for item in sorted(items, key=lambda value: (not value.must_keep, value.tokens)):
        if used + item.tokens <= budget or item.must_keep:
            kept.append(item)
            used += item.tokens
    return kept


context = [
    ContextItem('rules', '禁止重置用户代码', 80, True),
    ContextItem('memory', '天气 Skill 已迁移工具目录', 120, True),
    ContextItem('session', '当前计划：新增 Harness 教学目录', 160, True),
    ContextItem('chat', '早期闲聊内容', 500),
    ContextItem('tool-output', '很长的文件列表输出', 700),
]

[(item.layer, item.text) for item in compact_context(context, budget=450)]

## 第五原则：错误路径即主路径

编码代理运行中，错误不是偶发异常，而是常态：

- prompt 过长
- 输出超限
- 工具失败
- 文件被用户同时修改
- 中断发生在工具调用中间

可靠系统的目标不是“永不失败”，而是“失败以后能继续工作”。

In [ ]:
class RecoveryFuse:
    def __init__(self, max_attempts: int):
        self.max_attempts = max_attempts
        self.attempts = 0

    def can_retry(self) -> bool:
        return self.attempts < self.max_attempts

    def record_retry(self, action: str) -> dict:
        self.attempts += 1
        return {'attempt': self.attempts, 'action': action, 'remaining': self.max_attempts - self.attempts}


fuse = RecoveryFuse(max_attempts=3)
recovery_steps = []

for action in ['drain_pending_tool_result', 'compact_context', 'continue_generation', 'retry_again']:
    if not fuse.can_retry():
        recovery_steps.append({'action': action, 'status': 'blocked_by_fuse'})
        break
    recovery_steps.append(fuse.record_retry(action))

recovery_steps

## 第六原则：多代理靠分工隔离不确定性

多代理的价值不只是并发。

更重要的是隔离不确定性：

- 主线程负责协调和最终判断
- 研究 worker 负责读代码和归纳事实
- 实现 worker 负责限定范围内改代码
- 验证 worker 负责独立证明是否有效

验证必须独立，不能让实现者自评分。

In [ ]:
@dataclass
class AgentRole:
    name: str
    responsibility: str
    can_write: bool
    validates_own_work: bool


roles = [
    AgentRole('coordinator', '拆分任务并综合结果', False, False),
    AgentRole('research-worker', '阅读代码并回答具体问题', False, False),
    AgentRole('implementation-worker', '在限定文件内实现修改', True, False),
    AgentRole('verification-worker', '独立运行测试并报告风险', False, False),
]

[role for role in roles if role.validates_own_work]

上面最后一格应该返回空列表。

这表达了一个关键规则：**不允许系统自评分**。实现和验证要解耦。

## 第七原则：团队落地，制度优先于个人技巧

个人可以靠经验规避风险，团队不能靠每个人临场发挥。

团队要先明确：

1. 哪些目录可以改，哪些目录不能改
2. 哪些命令自动允许，哪些命令必须审批
3. 什么叫完成，必须跑哪些验证
4. 什么规则写进长期文件，什么只保留在会话里
5. 高频工作流是否要沉淀成 Skill
6. 可执行能力是否要沉淀成 Tool

制度稳定以后，再谈 Hook、审计、多代理自动化。

In [ ]:
team_policy = {
    'editable_paths': ['app/', 'docs/', 'notebooks/'],
    'forbidden_actions': ['git reset --hard', 'delete user changes'],
    'approval_required': ['destructive shell command', 'network dependency install'],
    'required_validation': ['syntax check', 'targeted tests', 'notebook json validation'],
    'memory_rule': 'stable rules go to AGENTS.md; temporary findings stay in session summary',
}

team_policy

## 九大设计原则总览

基于本 Notebook 的拆解，可以把 Claude Code 这类编码代理的设计原则归纳为九类：

1. Prompt 是控制面，不是人格装饰
2. Query Loop 是代理系统的心跳
3. 工具是受管执行接口，权限是基本器官
4. 上下文是工作内存，治理即预算治理
5. 错误路径即主路径，恢复目标是继续工作
6. 多代理靠分工隔离不确定性，验证必须独立
7. 团队落地：制度优先于个人技巧
8. 高危能力必须有高密度规则和审批分层
9. 叙事一致性很重要：系统要能解释自己为什么这么做

这些原则共同服务一个目标：让不稳定模型在稳定边界内工作。

## Harness Engineering 十条核心原则

把它压缩成十条，可以作为团队设计检查表：

1. 把模型当不稳定部件，而非可靠同事。
2. Prompt 是控制面，非人格包装。
3. Query Loop 是代理心跳，决定执行秩序。
4. 工具是受管接口，权限优先于能力。
5. 上下文是工作内存，分层治理、预算优先。
6. 错误是常态，恢复是主路径。
7. 恢复目标是继续工作，非礼貌收尾。
8. 多代理分区不确定性，分工大于并发。
9. 验证独立，不允许系统自评分。
10. 团队制度比个人技巧更重要。

## 落地清单

如果你要把个人 AI 编码工具升级成团队可复用系统，可以按这个顺序做：

1. 写清楚稳定规则：长期规则文件、项目边界、禁止动作
2. 定义工具权限：allow / deny / ask，而不是默认全放开
3. 定义上下文分层：规则、记忆、会话摘要、临时对话
4. 定义错误恢复：压缩、续写、重试、熔断
5. 定义验证标准：测试、lint、notebook 校验、人工复核
6. 再沉淀高频工作流：Skill 解决怎么做，Tool 解决能做什么
7. 最后再上 Hook、多代理、审计等高级能力

## 本阶段小结

这份 Notebook 的核心不是教你写一个更长的 Prompt。

它要建立一个更重要的工程判断：

```text
可靠性不来自模型承诺，而来自系统约束。
```

Prompt、Query Loop、Tool Permission、Context Compact、Recovery Fuse、Independent Verification，这些才是编码代理可靠性的主要来源。

下一步可以继续把这些原则映射回本仓库：比如设计一个最小 Query Loop，或者把天气助手智能体改造成可恢复、可验证、带权限裁决的版本。